# pyironflow

[`pyiron_workflow`](https://github.com/pyiron/pyiron_workflow) turns plain Python functions into the **nodes** of an explicit, inspectable workflow graph. Instead of relying on the order in which notebook cells happen to be executed, you declare how data flows between nodes, and `pyiron_workflow` figures out — and can cache — the rest. This notebook rebuilds the same [`workflow.py`](workflow.py) analysis from [`python.ipynb`](python.ipynb) — read CSV → convert load to stress/strain → fit Young's modulus → plot — as a graph.

In [1]:
from concurrent.futures import Future
from pyiron_workflow import Workflow, to_function_node
from pyironflow import PyironFlow
from workflow import (
    read_csv as _read_csv, 
    calculate_youngs_modulus as _calculate_youngs_modulus, 
    convert_load_to_stress as _convert_load_to_stress, 
    plot as _plot,
)

## Nodes

`to_function_node` wraps an existing Python function — unchanged — into a `pyiron_workflow` node. Each of the function's arguments becomes a node input, and its return value(s) become node output(s); nothing about `read_csv`, `convert_load_to_stress`, `calculate_youngs_modulus` or `plot` themselves had to change to make this work.

In [2]:
read_csv = to_function_node("read_csv", _read_csv, "read_csv")
calculate_youngs_modulus = to_function_node("calculate_youngs_modulus", _calculate_youngs_modulus, "calculate_youngs_modulus")
convert_load_to_stress = to_function_node("convert_load_to_stress", _convert_load_to_stress, "convert_load_to_stress")
plot = to_function_node("plot", _plot, "plot")

## Workflow

A `Workflow` is a container for nodes. Assigning `wf.<name> = node(...)` both adds the node to the graph and wires its inputs — either to a plain value or to another node's output, e.g. `wf.result_dict["stress"]`. Execution is lazy: nothing actually runs until `wf.run()` is called, at which point `pyiron_workflow` resolves the dependencies and executes the nodes in the right order.

In [3]:
wf = Workflow("my_workflow")

In [4]:
wf.area = 120
wf.strain_cutoff = 0.2
wf.filename = "./data/dataset_1.csv"
wf.df = read_csv(filename=wf.filename)
wf.result_dict = convert_load_to_stress(df=wf.df, area=wf.area)
wf.youngs_modulus = calculate_youngs_modulus(stress=wf.result_dict["stress"], strain=wf.result_dict["strain"], strain_cutoff=wf.strain_cutoff)
wf.plot = plot(stress=wf.result_dict["stress"], strain=wf.result_dict["strain"], format="-")

In [5]:
pf = PyironFlow(wf_list=[wf], root_path=".")
pf.gui

![image](images/vp_constructed.png)

## Reload Workflow

In [6]:
pf = PyironFlow(wf_list=[Workflow("workflow_reloaded")], root_path=".")
pf.gui

![image](images/vp_loading.png)